# 03 — Feature Engineering
## Peramalan Nilai PM2.5 Harian DKI Jakarta dengan SVR

Notebook ini membentuk fitur dari `data/pm25_daily.csv` (hasil `01_data_wrangling.ipynb`) berdasarkan kandidat lag yang ditemukan di `02_eda.ipynb`.

**Prinsip penting:** kandidat lag di sini sengaja dibuat **luas**, bukan dipersempit manual. Pemilihan fitur *final* dilakukan lewat forward selection + `TimeSeriesSplit` di `04_svr_modeling.ipynb`, berdasarkan performa model — bukan di notebook ini.

Langkah-langkah:
1. Muat `data/pm25_daily.csv`
2. Bentuk fitur lag `pm25` dari kandidat hasil notebook 02
3. Bentuk fitur rolling mean dan fitur musiman
4. Bentuk fitur lag-1 untuk polutan lain (dipakai di eksperimen notebook 04, bukan di model utama)
5. Buang baris yang tidak lengkap (`dropna`)
6. Split kronologis menjadi data latih dan data uji
7. Simpan `data/train.csv` dan `data/test.csv`

## 1. Setup dan Muat Data

In [1]:
import os

if 'google.colab' in str(get_ipython()) and not os.path.exists('data'):
    # Ganti USERNAME dengan username GitHub Anda
    !git clone https://github.com/YudhaAlfarizi/jakarta-pm25-svr-forecasting.git
    %cd jakarta-pm25-svr-forecasting

Cloning into 'jakarta-pm25-svr-forecasting'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 180 (delta 81), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 1.11 MiB | 6.05 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/jakarta-pm25-svr-forecasting


In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

IN_PATH = 'data/pm25_daily.csv'
OUT_TRAIN = 'data/train.csv'
OUT_TEST = 'data/test.csv'

# Tanggal batas split kronologis — ditetapkan sekali di sini,
# dipakai sama di notebook 04 (lihat docs/metodologi.md bagian 5)
TANGGAL_SPLIT = '2024-08-03'

In [3]:
df = pd.read_csv(IN_PATH, parse_dates=['tanggal']).set_index('tanggal').asfreq('D')

print(f"Jumlah baris: {len(df):,}")
print(f"Rentang     : {df.index.min().date()} s.d. {df.index.max().date()}")
df.head()

Jumlah baris: 1,520
Rentang     : 2021-01-01 s.d. 2025-02-28


,pm25,pm10,so2,co,o3,no2
tanggal,,,,,,
2021-01-01,58.0,38.0,2.0,11.0,65.0,6.0
2021-01-02,86.0,58.0,15.0,22.0,38.0,5.0
2021-01-03,93.0,64.0,14.0,20.0,35.0,5.0
2021-01-04,49.0,30.0,14.5,9.0,77.0,7.0
2021-01-05,89.0,59.0,15.0,19.0,42.0,7.0


## 2. Kandidat Lag dari Notebook 02

Hasil analisis PACF pada `02_eda.ipynb` (lag dengan `|PACF| > 1.96/sqrt(n)`, lihat bagian 8 pada notebook tersebut untuk tabel besaran lengkap):

In [4]:
# Kandidat lag diambil dari hasil notebook 02.
# Jika Anda menjalankan ulang notebook 02 dan hasilnya sedikit berbeda,
# perbarui daftar ini agar konsisten.
CANDIDATE_LAGS = [1, 2, 3, 4, 5, 6, 7, 8, 10, 14, 21]

print(f"Kandidat lag yang dipakai: {CANDIDATE_LAGS}")
print(f"Jumlah kandidat: {len(CANDIDATE_LAGS)}")

Kandidat lag yang dipakai: [1, 2, 3, 4, 5, 6, 7, 8, 10, 14, 21]
Jumlah kandidat: 11


## 3. Fitur Lag `pm25`

Setiap kandidat lag dibentuk sebagai kolom terpisah. Pemilihan subset final dilakukan di notebook 04, bukan di sini.

In [5]:
for lag in CANDIDATE_LAGS:
    df[f'pm25_lag{lag}'] = df['pm25'].shift(lag)

print("Kolom lag yang terbentuk:")
print([c for c in df.columns if c.startswith('pm25_lag')])

Kolom lag yang terbentuk:
['pm25_lag1', 'pm25_lag2', 'pm25_lag3', 'pm25_lag4', 'pm25_lag5', 'pm25_lag6', 'pm25_lag7', 'pm25_lag8', 'pm25_lag10', 'pm25_lag14', 'pm25_lag21']


## 4. Fitur Rolling Mean dan Fitur Musiman

Rolling mean memakai `shift(1)` agar nilai hari yang diprediksi tidak ikut masuk ke fiturnya sendiri (lihat `docs/metodologi.md` bagian 4).

In [6]:
df['pm25_roll7'] = df['pm25'].shift(1).rolling(7).mean()
df['pm25_roll14'] = df['pm25'].shift(1).rolling(14).mean()

df['month'] = df.index.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df = df.drop(columns=['month'])

print("Fitur tambahan:", ['pm25_roll7', 'pm25_roll14', 'month_sin', 'month_cos'])

Fitur tambahan: ['pm25_roll7', 'pm25_roll14', 'month_sin', 'month_cos']


## 5. Fitur Lag-1 Polutan Lain (untuk eksperimen notebook 04)

Fitur ini **tidak dipakai di model utama**, tetapi disediakan di sini supaya notebook 04 bisa menguji apakah menambah polutan lain memperbaiki akurasi. Semua memakai `shift(1)` karena nilai polutan hari ini belum diketahui saat meramalkan hari itu.

In [7]:
for kolom in ['pm10', 'so2', 'co', 'o3', 'no2']:
    df[f'{kolom}_lag1'] = df[kolom].shift(1)

print("Kolom lag-1 polutan:", [c for c in df.columns if c.endswith('_lag1') and not c.startswith('pm25')])

Kolom lag-1 polutan: ['pm10_lag1', 'so2_lag1', 'co_lag1', 'o3_lag1', 'no2_lag1']


## 6. Membuang Baris Tidak Lengkap

Baris di awal data (karena butuh riwayat sampai 21 hari ke belakang) dan baris yang melewati celah panjang (mis. `pm10` yang punya celah panjang di 2023, lihat `docs/dataset.md` bagian 6) akan otomatis terbuang di sini.

In [8]:
kolom_target = ['pm25']
kolom_fitur_pm25 = [c for c in df.columns if c.startswith('pm25_lag') or c in ('pm25_roll7', 'pm25_roll14', 'month_sin', 'month_cos')]
kolom_fitur_polutan = [c for c in df.columns if c.endswith('_lag1') and not c.startswith('pm25')]

semua_kolom_dipakai = kolom_target + kolom_fitur_pm25 + kolom_fitur_polutan

print(f"Baris sebelum dropna: {len(df):,}")
df_final = df[semua_kolom_dipakai].dropna(subset=kolom_target + kolom_fitur_pm25)
print(f"Baris setelah dropna (hanya mewajibkan fitur pm25 lengkap): {len(df_final):,}")
print(f"\nCatatan: kolom polutan ({kolom_fitur_polutan}) BOLEH tetap mengandung NaN")
print("di sini, karena baru dibutuhkan saat notebook 04 menjalankan eksperimen")
print("yang memakainya secara spesifik.")

Baris sebelum dropna: 1,520
Baris setelah dropna (hanya mewajibkan fitur pm25 lengkap): 1,499

Catatan: kolom polutan (['pm10_lag1', 'so2_lag1', 'co_lag1', 'o3_lag1', 'no2_lag1']) BOLEH tetap mengandung NaN
di sini, karena baru dibutuhkan saat notebook 04 menjalankan eksperimen
yang memakainya secara spesifik.


## 7. Split Kronologis

In [9]:
split_date = pd.Timestamp(TANGGAL_SPLIT)

train = df_final[df_final.index < split_date]
test = df_final[df_final.index >= split_date]

print(f"Data latih: {train.index.min().date()} s.d. {train.index.max().date()} ({len(train):,} baris)")
print(f"Data uji  : {test.index.min().date()} s.d. {test.index.max().date()} ({len(test):,} baris)")
print(f"Rasio     : {len(train)/(len(train)+len(test))*100:.1f}% latih, {len(test)/(len(train)+len(test))*100:.1f}% uji")

Data latih: 2021-01-22 s.d. 2024-08-02 (1,289 baris)
Data uji  : 2024-08-03 s.d. 2025-02-28 (210 baris)
Rasio     : 86.0% latih, 14.0% uji


## 8. Menyimpan Hasil

In [10]:
train.to_csv(OUT_TRAIN, index=True, index_label='tanggal')
test.to_csv(OUT_TEST, index=True, index_label='tanggal')

print(f"Tersimpan: {OUT_TRAIN} ({len(train)} baris, {train.shape[1]} kolom)")
print(f"Tersimpan: {OUT_TEST} ({len(test)} baris, {test.shape[1]} kolom)")
train.head()

Tersimpan: data/train.csv (1289 baris, 21 kolom)
Tersimpan: data/test.csv (210 baris, 21 kolom)


,pm25,pm25_lag1,pm25_lag2,pm25_lag3,pm25_lag4,pm25_lag5,pm25_lag6,pm25_lag7,pm25_lag8,pm25_lag10,pm25_lag14,pm25_lag21,pm25_roll7,pm25_roll14,month_sin,month_cos,pm10_lag1,so2_lag1,co_lag1,o3_lag1,no2_lag1
tanggal,,,,,,,,,,,,,,,,,,,,,
2021-01-22,112.0,44.0,45.0,42.0,49.0,75.0,92.0,82.0,78.0,51.0,55.0,58.0,61.285714,59.642857,0.5,0.866025,46.0,12.0,9.0,67.0,13.0
2021-01-23,126.0,112.0,44.0,45.0,42.0,49.0,75.0,92.0,82.0,44.0,67.0,86.0,65.571429,63.714286,0.5,0.866025,84.0,20.0,33.0,57.0,10.0
2021-01-24,90.0,126.0,112.0,44.0,45.0,42.0,49.0,75.0,92.0,78.0,39.0,93.0,70.428571,67.928571,0.5,0.866025,89.0,19.0,42.0,64.0,13.0
2021-01-25,86.0,90.0,126.0,112.0,44.0,45.0,42.0,49.0,75.0,82.0,72.0,49.0,72.571429,71.571429,0.5,0.866025,64.0,19.0,31.0,34.0,10.0
2021-01-26,79.0,86.0,90.0,126.0,112.0,44.0,45.0,42.0,49.0,92.0,51.0,89.0,77.857143,72.571429,0.5,0.866025,62.0,14.0,26.0,134.0,14.0


## Ringkasan

| Aspek | Hasil |
|---|---|
| Kandidat lag `pm25` | `[1, 2, 3, 4, 5, 6, 7, 8, 10, 14, 21]` — dari PACF notebook 02, belum final |
| Fitur tambahan | `pm25_roll7`, `pm25_roll14`, `month_sin`, `month_cos` |
| Fitur eksperimen (notebook 04) | `pm10_lag1`, `so2_lag1`, `co_lag1`, `o3_lag1`, `no2_lag1` |
| Split | Kronologis, batas `2024-08-03` |
| Output | `data/train.csv`, `data/test.csv` |

**Penting:** fitur final yang benar-benar dipakai model utama belum ditentukan di sini. Itu akan diputuskan di `04_svr_modeling.ipynb` lewat forward selection berbasis performa, bukan oleh notebook ini.

**Langkah berikutnya:** buka `04_svr_modeling.ipynb`.